# Data Loaders

In [ ]:
import os
import cv2
import numpy as np
from torch.utils.data import Dataset, DataLoader

class PotholeDataset(Dataset):
    def __init__(self, image_dir, label_dir):
        self.image_dir = image_dir
        self.label_dir = label_dir
        self.image_files = [f for f in os.listdir(image_dir) if f.endswith('.jpg')]

    def __len__(self):
        return len(self.image_files)

    def __getitem__(self, idx):
        image_name = self.image_files[idx]
        img_path = os.path.join(self.image_dir, image_name)
        image = cv2.imread(img_path)
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

        # normalize
        image = image.astype(np.float32) / 255.0

        # create empty mask
        mask = np.zeros(image.shape[:-1], dtype=np.float32)

        label_path = os.path.join(self.label_dir, image_name.replace('.jpg', '.txt'))
        with open(label_path, 'r') as f:
            lines = f.readlines()
            for l in lines:
                parts = l.strip().split()
                class_id = int(parts[0])
                coords = np.array([float(x) for x in parts[1:]]).reshape(-1, 2)

                coords[:, 0] *= image.shape[0]
                coords[:, 1] *= image.shape[0]
                coords = coords.astype(np.int32)
                cv2.fillPoly(mask, [coords], color=1.0)
        mask = np.expand_dims(mask, axis=-1)           
        return image, mask

        

In [ ]:
def getPotholeDataloader(batch_size, image_dir, label_dir):
    dataset = PotholeDataset(image_dir, label_dir)     
    def numpy_collate(batch):
        images, masks = zip(*batch)
        return np.stack(images), np.stack(masks)
        
    return DataLoader(
        dataset, batch_size=batch_size, shuffle=True, 
        drop_last=True, collate_fn=numpy_collate, num_workers=4
    )

In [ ]:
# Test
dataset = PotholeDataset("/kaggle/input/datasets/farzadnekouei/pothole-image-segmentation-dataset/Pothole_Segmentation_YOLOv8/train/images", "/kaggle/input/datasets/farzadnekouei/pothole-image-segmentation-dataset/Pothole_Segmentation_YOLOv8/train/labels")     
loader = getPotholeDataloader(4, 
                     "/kaggle/input/datasets/farzadnekouei/pothole-image-segmentation-dataset/Pothole_Segmentation_YOLOv8/train/images", 
                     "/kaggle/input/datasets/farzadnekouei/pothole-image-segmentation-dataset/Pothole_Segmentation_YOLOv8/train/labels")
dataset[0]

# Rotary Position Embeddings (RoPE)

In [ ]:
import jax
import jax.numpy as jnp 
import flax.linen as nn

def apply_rope(x, head_dim):
    # X -> B,S,H,head_dim
    seq_len = x.shape[1]
    num_heads = x.shape[2]
    dims = jnp.arange(seq_len, dtype=jnp.float32).reshape(-1, 1) # seq_len, 1
    i_2 = jnp.arange(0, head_dim, 2, dtype=jnp.float32)
    div_term = jnp.exp(-1* i_2 * jnp.log(100000)/head_dim) # head_dim//2
    freq = dims * div_term # seq_len, head_dim//2
    sin_freq = jnp.repeat(jnp.sin(freq), 2, axis=-1).reshape(1, seq_len, 1, head_dim)
    cos_freq = jnp.repeat(jnp.cos(freq), 2, axis=-1).reshape(1, seq_len, 1, head_dim)
    odd_terms = x[..., 1::2]
    even_terms = x[..., 0::2]
    x_tmp = jnp.stack([-odd_terms, even_terms], axis=-1).reshape(x.shape)
    return x * cos_freq + x_tmp * sin_freq


In [2]:
import jax
import jax.numpy as jnp 
import flax.linen as nn

def apply_rope2d(x, grid_h, grid_w):
    B, S, num_heads, head_dim = x.shape
    w_indices = jnp.arange(grid_w, dtype=jnp.float32)
    h_indices = jnp.arange(grid_h, dtype=jnp.float32)
    grid_col, grid_row = jnp.meshgrid(w_indices, h_indices)
    
    flat_row = grid_row.reshape(-1, 1) # seq_len, 1
    flat_col = grid_col.reshape(-1, 1) # seq_len, 1

    i_2 = jnp.arange(0, head_dim//2, 2, dtype=jnp.float32) // (head_dim//2)
    div_term = jnp.exp(-1 * i_2 * jnp.log(100000)) # head_dim//4
    
    sin_row = jnp.repeat(jnp.sin(flat_row * div_term), 2, axis=-1).reshape(1, seq_len, 1, head_dim//2) # seq_len, head_dim//2
    cos_row = jnp.repeat(jnp.cos(flat_row * div_term), 2, axis=-1).reshape(1, seq_len, 1, head_dim//2) # seq_len, head_dim//2

    sin_col = jnp.repeat(jnp.sin(flat_col * div_term), 2, axis=-1).reshape(1, seq_len, 1, head_dim//2) # seq_len, head_dim//2
    cos_col = jnp.repeat(jnp.cos(flat_col * div_term), 2, axis=-1).reshape(1, seq_len, 1, head_dim//2) # seq_len, head_dim//2

    sin = jnp.stack(sin_row, sin_col, axis=-1)
    cos = jnp.stack(cos_row, cos_col, axis=-1)

    x_tmp = jnp.stack(-x[...,1::2], x[...,0::2], axis=-1).reshape(x.shape)
    return x * cos + x_tmp * sin

In [ ]:
# test
key = jax.random.key(42)
apply_rope(jax.random.normal(key, shape=(5, 10, 12, 20)), 20)

In [ ]:
# test
key = jax.random.key(42)
apply_rope_2d(jax.random.normal(key, shape=(5, 10, 12, 20)), 2, 5)

# Multi-Head Self-Attention 

In [ ]:
class MHSA(nn.Module):
    num_heads: int
    
    @nn.compact
    def __call__(self, x):
        # x -> Batch, Seq_len, dim
        b, s, dim = x.shape
        head_dim = dim//self.num_heads

        w_q = self.param('w_q', nn.initializers.lecun_normal(), (dim, dim))
        w_k = self.param('w_k', nn.initializers.lecun_normal(), (dim, dim))
        w_v = self.param('w_v', nn.initializers.lecun_normal(), (dim, dim))
        w_o = self.param('w_o', nn.initializers.lecun_normal(), (dim, dim))

        q = jnp.dot(x, w_q).reshape(b, s, self.num_heads, head_dim)
        k = jnp.dot(x, w_k).reshape(b, s, self.num_heads, head_dim)
        v = jnp.dot(x, w_v).reshape(b, s, self.num_heads, head_dim)

        q = apply_rope(q, head_dim)
        k = apply_rope(k, head_dim)
        
        qk_t = jnp.einsum("bqhd, bkhd -> bhqk", q, k) 
        atten_scores = qk_t/jnp.sqrt(head_dim)
        attention = jax.nn.softmax(atten_scores, axis=-1)
 
        att_out = jnp.einsum('bhqk, bkhd -> bqhd', attention, v).reshape(b, s, dim)
        
        out = jnp.dot(att_out, w_o)
        return out

# Transformer block and adaLN

In [ ]:
class AdaLN(nn.Module):
    dim: int

    @nn.compact
    def __call__(self, x, cond):
        # Zero initializers are important because when we start training we have no idea on scale, shift and gate, so it'll behave as a passthrough
        mlp_out = nn.Dense(self.dim*3, kernel_init=nn.initializers.zeros, bias_init=nn.initializers.zeros)(cond)
        shift, scale, gate = jnp.split(mlp_out, 3, axis=-1)
        x = nn.LayerNorm()(x)

        x = x * (1 + jnp.expand_dims(scale, axis = 1)) + jnp.expand_dims(shift, axis = 1)

        return x, jnp.expand_dims(gate, axis = 1)

In [ ]:
class TransformerBlock(nn.Module):
    num_heads: int
    
    @nn.compact
    def __call__(self, x, global_context):
        # attention path
        dim = x.shape[-1]
        x_norm, mhsa_gate = AdaLN(dim)(x, global_context)
        mhsa_out = MHSA(self.num_heads)(x_norm)        
        x = x + mhsa_gate * mhsa_out

        # MLP path
        x_norm_mlp, mlp_gate = AdaLN(dim)(x, global_context)
        mlp_out = nn.Dense(dim*4)(x_norm_mlp)
        mlp_out = nn.gelu(mlp_out)
        mlp_out = nn.Dense(dim)(mlp_out)
        x = x + mlp_gate * mlp_out
        
        return x      

# Model

In [ ]:
class PotholeSegmenter(nn.Module):
    emb_dim: int = 256
    patch_size: int = 16
    depth: int = 6
    num_heads: int = 8
    
    @nn.compact
    def __call__(self, images):
        B,H,W,C = images.shape
        grid_h = H//self.patch_size
        grid_w = W//self.patch_size

        x = images.reshape((B, grid_h, self.patch_size, grid_w, self.patch_size, C))
        x = jnp.einsum('abcdef -> abdcef',x)
        x = x.reshape((B, grid_h*grid_w, -1))
        
        x = nn.Dense(self.emb_dim)(x)
        global_context = jnp.mean(x, axis=1)

        for _ in range(self.depth):
            x = TransformerBlock(self.num_heads)(x, global_context)

        x = nn.LayerNorm()(x)
        x = nn.Dense(self.patch_size*self.patch_size*1)(x)
        x = x.reshape(B, grid_h, grid_w, self.patch_size, self.patch_size, 1)
        x = jnp.einsum('ahwpqr -> ahpwqr', x)
        logits = x.reshape((B, H, W, 1))
        return logits

# Training 


In [ ]:
import optax
from flax.training import train_state
from jax.sharding import Mesh, PartitionSpec, NamedSharding
from jax.experimental import mesh_utils
from functools import partial
import orbax.checkpoint as ocp

In [ ]:
def bce_with_dice_loss(logits, target):
    bce = optax.sigmoid_binary_cross_entropy(logits, target)
    probs = jax.nn.sigmoid(logits)
    intersection = jnp.sum(probs*target, axis=(1,2,3))
    union = jnp.sum(probs, axis=(1,2,3)) + jnp.sum(target, axis=(1,2,3))
    iou = (2*intersection+1e-5)/(union + 1e-5)
    dice_loss = 1 - jnp.mean(iou)
    return jnp.mean(bce) + dice_loss

In [ ]:
num_devices = jax.device_count()
print(num_devices)
devices = mesh_utils.create_device_mesh((num_devices,))
mesh = Mesh(devices, axis_names=('batch',))
print(mesh)

data_sharding = NamedSharding(mesh, PartitionSpec('batch', None, None, None))
param_sharding = NamedSharding(mesh, PartitionSpec())

@partial(
    jax.jit,
    in_shardings=(param_sharding, data_sharding, data_sharding),
    out_shardings=(param_sharding, param_sharding) 
)
def train_step(state, images, masks):
    def loss_fn(params):
        logits = state.apply_fn({'params': params}, images)
        loss = bce_with_dice_loss(logits, masks)
        return loss
        
    grad_fn = jax.value_and_grad(loss_fn)
    loss, grads = grad_fn(state.params)
    new_state = state.apply_gradients(grads=grads)
    return new_state, loss

def train_model():
    batch_size = 16 
    train_loader = getPotholeDataloader(batch_size, 
                     "/kaggle/input/datasets/farzadnekouei/pothole-image-segmentation-dataset/Pothole_Segmentation_YOLOv8/train/images", 
                     "/kaggle/input/datasets/farzadnekouei/pothole-image-segmentation-dataset/Pothole_Segmentation_YOLOv8/train/labels")
    
    rng = jax.random.PRNGKey(42)
    model = PotholeSegmenter()
    dummy_x = jnp.ones((1, 640, 640, 3))
    
    variables = model.init(rng, dummy_x)
    
    total_steps = len(train_loader) * 50 
    lr_schedule = optax.cosine_decay_schedule(init_value=1e-4, decay_steps=total_steps)
    tx = optax.adamw(learning_rate=lr_schedule, weight_decay=1e-4)
    
    state = train_state.TrainState.create(
        apply_fn=model.apply,
        params=variables['params'],
        tx=tx
    )

    #Orbax Checkpointer Setup in Kaggle Working Directory
    checkpoint_dir = os.path.abspath('/kaggle/working/checkpoints')
    checkpointer = ocp.StandardCheckpointer()
    options = ocp.CheckpointManagerOptions(max_to_keep=3, create=True)
    mngr = ocp.CheckpointManager(checkpoint_dir, checkpointer, options)
    
    epochs = 50
    for epoch in range(epochs):
        epoch_loss = 0.0
        
        for batch_idx, (images_np, masks_np) in enumerate(train_loader):
            images = jax.device_put(images_np, data_sharding)
            masks = jax.device_put(masks_np, data_sharding)
            
            state, loss = train_step(state, images, masks)
            epoch_loss += loss
            
            if batch_idx % 10 == 0:
                print(f"Epoch {epoch+1}, Batch {batch_idx}, Loss: {loss:.4f}")
                
        print(f"--- Epoch {epoch+1} Complete. Avg Loss: {epoch_loss / len(train_loader):.4f} ---")
        
        # Save checkpoint at the end of each epoch to /kaggle/working/checkpoints/
        mngr.save(epoch, args=ocp.args.StandardSave(state))
        mngr.wait_until_finished()
        print(f"Checkpoint saved for epoch {epoch+1} at {checkpoint_dir}")

if __name__ == "__main__":
    train_model()


# Evals

In [ ]:
import os
import cv2
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import jax
import jax.numpy as jnp
import orbax.checkpoint as ocp
from flax.training import train_state
import optax
import random



def get_val_dataloader(batch_size, image_dir, label_dir):
    dataset = PotholeDataset(image_dir, label_dir)

    def numpy_collate(batch):
        images, masks = zip(*batch)
        return np.stack(images), np.stack(masks)

    return dataset, dataset.image_files, numpy_collate


# ---------------------------------------------------------------------------
# Metric Calculations (NumPy / CPU execution for stability across splits)
# ---------------------------------------------------------------------------
def calculate_metrics(pred_masks, true_masks, threshold=0.5, eps=1e-7):
    """
    pred_masks: [B, H, W, 1] sigmoid probabilities
    true_masks: [B, H, W, 1] binary target values (0 or 1)
    """
    preds_binary = (pred_masks > threshold).astype(np.float32)
    targets_binary = (true_masks > 0.5).astype(np.float32)

    # Flatten spatial axes: [B, H*W]
    preds_flat = preds_binary.reshape(preds_binary.shape[0], -1)
    targets_flat = targets_binary.reshape(targets_binary.shape[0], -1)

    intersection = np.sum(preds_flat * targets_flat, axis=-1)
    total_area = np.sum(preds_flat, axis=-1) + np.sum(targets_flat, axis=-1)
    union = total_area - intersection

    # IoU = Intersection / Union
    # Handle empty ground-truth edge case (true negatives)
    both_empty = (np.sum(targets_flat, axis=-1) == 0) & (np.sum(preds_flat, axis=-1) == 0)
    
    iou = (intersection + eps) / (union + eps)
    dice = (2.0 * intersection + eps) / (total_area + eps)

    # If both ground truth and prediction are blank, score as perfect 1.0
    iou[both_empty] = 1.0
    dice[both_empty] = 1.0

    return iou, dice


# ---------------------------------------------------------------------------
# JIT Inference Step
# ---------------------------------------------------------------------------
@jax.jit
def eval_step(params, images):
    model = PotholeSegmenter()
    logits = model.apply({'params': params}, images)
    probs = jax.nn.sigmoid(logits)
    return probs


# ---------------------------------------------------------------------------
# Visual Inspection Utility for Quick Sample Display
# ---------------------------------------------------------------------------
def visualize_val_samples(
    trained_model_or_params,
    num_samples=4,
    random_selection=True,
    threshold=0.5,
    val_image_dir='/kaggle/input/pothole-segmentation/valid/images',
    val_label_dir='/kaggle/input/pothole-segmentation/valid/labels'
):
    """
    Renders clean, high-resolution side-by-side comparisons of:
      1. Raw Input Image
      2. Target (Ground Truth Mask)
      3. Model Output (Continuous Sigmoid Probability Heatmap)
      4. Prediction Overlay (Mask over original image with IoU & Dice metrics)
    """
    if hasattr(trained_model_or_params, 'params'):
        params = trained_model_or_params.params
    elif trained_model_or_params is not None:
        params = trained_model_or_params
    else:
        raise ValueError("Please provide `state` or `state.params` to visualize samples.")

    dataset = PotholeDataset(val_image_dir, val_label_dir)
    total_val = len(dataset)
    num_samples = min(num_samples, total_val)

    if random_selection:
        indices = random.sample(range(total_val), num_samples)
    else:
        indices = list(range(num_samples))

    print(f"Displaying {num_samples} validation samples (Threshold: {threshold})...\n")

    for i, idx in enumerate(indices):
        img, true_mask = dataset[idx]
        img_name = dataset.image_files[idx]

        # Forward pass on single image
        img_batch = jnp.expand_dims(jnp.array(img), axis=0)
        pred_prob = eval_step(params, img_batch)
        pred_prob_np = np.array(pred_prob)[0, :, :, 0]  # [H, W]
        target_mask_np = true_mask[:, :, 0]             # [H, W]

        iou, dice = calculate_metrics(
            pred_prob_np[np.newaxis, :, :, np.newaxis],
            true_mask[np.newaxis, ...]
        )
        sample_iou = iou[0]
        sample_dice = dice[0]

        pred_binary = (pred_prob_np >= threshold).astype(np.float32)

        overlay = img.copy()
        red_tint = np.array([1.0, 0.15, 0.15], dtype=np.float32)
        mask_expanded = pred_binary[:, :, np.newaxis]
        alpha = 0.5
        overlay = np.where(mask_expanded == 1.0, (1 - alpha) * overlay + alpha * red_tint, overlay)

        fig, axes = plt.subplots(1, 4, figsize=(20, 5))

        axes[0].imshow(img)
        axes[0].set_title(f"Sample {i+1} | Input: {img_name}", fontsize=11, fontweight='semibold')
        axes[0].axis('off')

        axes[1].imshow(target_mask_np, cmap='gray')
        axes[1].set_title(f"Target (Ground Truth)\nPothole Pixels: {int(np.sum(target_mask_np))}", fontsize=11, fontweight='semibold')
        axes[1].axis('off')

        im_heat = axes[2].imshow(pred_prob_np, cmap='magma', vmin=0.0, vmax=1.0)
        axes[2].set_title(f"Pred Output (Confidence)\nPeak Prob: {pred_prob_np.max():.2f}", fontsize=11, fontweight='semibold')
        axes[2].axis('off')
        plt.colorbar(im_heat, ax=axes[2], fraction=0.046, pad=0.04)

        axes[3].imshow(overlay)
        axes[3].set_title(f"Prediction Overlay (T={threshold})\nIoU: {sample_iou * 100:.2f}% | Dice: {sample_dice * 100:.2f}%", 
                          fontsize=11, fontweight='semibold',
                          color='darkgreen' if sample_iou > 0.4 else 'darkred')
        axes[3].axis('off')

        plt.tight_layout()
        plt.show()
        plt.close(fig)


# ---------------------------------------------------------------------------
# Robust Checkpoint Loader Helper
# ---------------------------------------------------------------------------
def load_checkpoint_params(checkpoint_dir, dummy_state):
    """
    Restores model parameters from the saved Orbax structure.
    Targets latest step (e.g. step 49) and navigates the 'default' item folder.
    """
    checkpoint_dir = Path(checkpoint_dir)
    if not checkpoint_dir.exists():
        raise FileNotFoundError(f"Checkpoint directory does not exist: {checkpoint_dir}")

    # Discover latest step folder
    numeric_steps = []
    for p in checkpoint_dir.iterdir():
        if p.is_dir() and p.name.isdigit():
            numeric_steps.append(int(p.name))
            
    if not numeric_steps:
        raise FileNotFoundError(f"No checkpoint folders found in {checkpoint_dir}")
        
    latest_step = max(numeric_steps)
    step_dir = checkpoint_dir / str(latest_step)
    print(f"Targeting checkpoint step: {latest_step} at {step_dir}")

    # CheckpointManager with StandardRestore
    try:
        mngr = ocp.CheckpointManager(checkpoint_dir)
        restored = mngr.restore(latest_step, args=ocp.args.StandardRestore(dummy_state))
        if hasattr(restored, 'params'):
            return restored.params
        if isinstance(restored, dict) and 'params' in restored:
            return restored['params']
    except Exception as e:
        print(f"CheckpointManager restore note: {e}. Trying direct checkpointer...")

    # Direct StandardCheckpointer on step_dir/default
    default_dir = step_dir / "default"
    target_path = default_dir if default_dir.exists() else step_dir
    print(f"Loading via StandardCheckpointer from: {target_path}")

    checkpointer = ocp.StandardCheckpointer()
    try:
        restored = checkpointer.restore(target_path, item=dummy_state)
        if hasattr(restored, 'params'):
            return restored.params
        if isinstance(restored, dict) and 'params' in restored:
            return restored['params']
    except Exception as e2:
        print(f"Typed checkpointer restore failed: {e2}. Trying untyped PyTreeCheckpointer...")

    # Fallback: PyTreeCheckpointer without schema enforcement
    pytree_cp = ocp.PyTreeCheckpointer()
    restored = pytree_cp.restore(target_path)
    if hasattr(restored, 'params'):
        return restored.params
    if isinstance(restored, dict):
        if 'params' in restored:
            return restored['params']
        if 'state' in restored and 'params' in restored['state']:
            return restored['state']['params']

    raise RuntimeError(f"Unable to extract params from {target_path}")


# ---------------------------------------------------------------------------
# Main Evaluation Routine
# ---------------------------------------------------------------------------
def run_evaluation(trained_model_or_params=None):
    """
    Evaluates the model on the validation split.
    """
    val_image_dir = '/kaggle/input/datasets/farzadnekouei/pothole-image-segmentation-dataset/Pothole_Segmentation_YOLOv8/valid/images'
    val_label_dir = '/kaggle/input/datasets/farzadnekouei/pothole-image-segmentation-dataset/Pothole_Segmentation_YOLOv8/valid/labels'
    checkpoint_dir = os.path.abspath('/kaggle/working/checkpoints')
    output_vis_dir = '/kaggle/working/val_predictions'
    os.makedirs(output_vis_dir, exist_ok=True)

    # 1. Resolve weights
    if trained_model_or_params is not None:
        print("Using model directly from memory...")
        if hasattr(trained_model_or_params, 'params'):
            trained_params = trained_model_or_params.params
        else:
            trained_params = trained_model_or_params
    else:
        print("Restoring checkpoint from disk...")
        model = PotholeSegmenter()
        dummy_x = jnp.ones((1, 640, 640, 3), dtype=jnp.float32)
        variables = model.init(jax.random.PRNGKey(0), dummy_x)
        
        tx = optax.adamw(learning_rate=1e-4, weight_decay=1e-4)
        dummy_state = train_state.TrainState.create(
            apply_fn=model.apply,
            params=variables['params'],
            tx=tx
        )

        trained_params = load_checkpoint_params(checkpoint_dir, dummy_state)
        print("Successfully restored parameters from checkpoint.")

    # 2. Load Validation Dataset
    dataset = PotholeDataset(val_image_dir, val_label_dir)
    print(f"Total validation samples: {len(dataset)}")

    all_ious = []
    all_dices = []
    all_pred_probs = []
    all_gt_masks = []
    
    batch_size = 8
    num_samples = len(dataset)
    saved_visualizations = 0

    print("Running evaluation across validation dataset...")
    for start_idx in range(0, num_samples, batch_size):
        end_idx = min(start_idx + batch_size, num_samples)
        
        batch_images = []
        batch_masks = []
        batch_names = []

        for idx in range(start_idx, end_idx):
            img, mask = dataset[idx]
            batch_images.append(img)
            batch_masks.append(mask)
            batch_names.append(dataset.image_files[idx])

        batch_images = np.stack(batch_images)
        batch_masks = np.stack(batch_masks)

        pred_probs = eval_step(trained_params, jnp.array(batch_images))
        pred_probs_np = np.array(pred_probs)

        # Store for dataset-wide threshold analysis
        all_pred_probs.append(pred_probs_np)
        all_gt_masks.append(batch_masks)

        ious, dices = calculate_metrics(pred_probs_np, batch_masks)
        all_ious.extend(ious.tolist())
        all_dices.extend(dices.tolist())

        for b in range(end_idx - start_idx):
            if saved_visualizations < 10:
                fig, axes = plt.subplots(1, 4, figsize=(18, 5))
                
                axes[0].imshow(batch_images[b])
                axes[0].set_title(f"Sample {saved_visualizations+1}: Input Image", fontsize=11)
                axes[0].axis('off')

                axes[1].imshow(batch_masks[b, :, :, 0], cmap='gray')
                axes[1].set_title("Ground Truth Mask", fontsize=11)
                axes[1].axis('off')

                im = axes[2].imshow(pred_probs_np[b, :, :, 0], cmap='magma', vmin=0.0, vmax=1.0)
                axes[2].set_title("Confidence Heatmap", fontsize=11)
                axes[2].axis('off')
                plt.colorbar(im, ax=axes[2], fraction=0.046, pad=0.04)

                bin_mask = (pred_probs_np[b, :, :, 0] > 0.5).astype(np.float32)
                overlay = batch_images[b].copy()
                overlay[bin_mask == 1.0] = [1.0, 0.2, 0.2]
                
                axes[3].imshow(overlay)
                axes[3].set_title(f"Pred Overlay\nIoU: {ious[b]:.3f} | Dice: {dices[b]:.3f}", fontsize=11)
                axes[3].axis('off')

                save_path = os.path.join(output_vis_dir, f"val_{saved_visualizations:02d}_{batch_names[b]}")
                plt.tight_layout()
                plt.savefig(save_path, bbox_inches='tight', dpi=150)
                plt.show()
                plt.close(fig)
                
                saved_visualizations += 1

    all_ious = np.array(all_ious)
    all_dices = np.array(all_dices)
    full_val_preds = np.concatenate(all_pred_probs, axis=0)
    full_val_masks = np.concatenate(all_gt_masks, axis=0)

    mean_iou = np.mean(all_ious)
    mean_dice = np.mean(all_dices)
    median_iou = np.median(all_ious)
    median_dice = np.median(all_dices)

    print("\n" + "=" * 55)
    print("             VALIDATION BENCHMARK RESULTS           ")
    print("=" * 55)
    print(f"Evaluated Samples : {num_samples}")
    print(f"Mean IoU          : {mean_iou * 100:.2f}%")
    print(f"Mean Dice (F1)    : {mean_dice * 100:.2f}%")
    print(f"Median IoU        : {median_iou * 100:.2f}%")
    print(f"Median Dice (F1)  : {median_dice * 100:.2f}%")
    print("=" * 55)
    print(f"Visualization plots saved to: {output_vis_dir}")

    # Full Dataset Threshold Sensitivity Sweep
    print("\nThreshold Sensitivity Sweep (Dataset-wide IoU vs Decision Boundary):")
    for t in [0.20, 0.30, 0.40, 0.50, 0.60]:
        t_iou, t_dice = calculate_metrics(full_val_preds, full_val_masks, threshold=t)
        print(f"  T = {t:.2f} -> Dataset IoU: {np.mean(t_iou)*100:.2f}% | Dataset Dice: {np.mean(t_dice)*100:.2f}%")

    fig, axes = plt.subplots(1, 2, figsize=(14, 4))
    
    axes[0].hist(all_ious * 100, bins=20, color='royalblue', edgecolor='black', alpha=0.75)
    axes[0].axvline(mean_iou * 100, color='red', linestyle='--', label=f'Mean: {mean_iou * 100:.2f}%')
    axes[0].axvline(median_iou * 100, color='green', linestyle=':', label=f'Median: {median_iou * 100:.2f}%')
    axes[0].set_title("IoU Distribution Across Validation Set", fontsize=12)
    axes[0].set_xlabel("IoU (%)")
    axes[0].set_ylabel("Count")
    axes[0].legend()
    axes[0].grid(axis='y', linestyle='--', alpha=0.5)

    axes[1].hist(all_dices * 100, bins=20, color='darkorange', edgecolor='black', alpha=0.75)
    axes[1].axvline(mean_dice * 100, color='red', linestyle='--', label=f'Mean: {mean_dice * 100:.2f}%')
    axes[1].axvline(median_dice * 100, color='green', linestyle=':', label=f'Median: {median_dice * 100:.2f}%')
    axes[1].set_title("Dice (F1) Score Distribution", fontsize=12)
    axes[1].set_xlabel("Dice Score (%)")
    axes[1].set_ylabel("Count")
    axes[1].legend()
    axes[1].grid(axis='y', linestyle='--', alpha=0.5)

    plt.tight_layout()
    plt.savefig(os.path.join(output_vis_dir, "metrics_distribution.png"), bbox_inches='tight', dpi=150)
    plt.show()
    plt.close(fig)


if __name__ == '__main__':
    run_evaluation()